# FerretNet: AI vs Real Image Detection — Full Pipeline

This notebook runs the entire pipeline:
1. **Setup** — Verify imports and model loading
2. **Data** — Download GRAVEX-200K + precompute face crops
3. **Augmentation Report** — Visualize each augmentation with before/after figures
4. **Training** — Fine-tune DualBranchFerretNet
5. **Evaluation** — Test set metrics and ROC curves
6. **Inference** — Run on sample images

## 1. Setup & Verification

In [1]:
import sys
sys.path.insert(0, '/Users/MAC/Desktop/Projects/AI_COMPUTER_VISION')

import torch
import torch.multiprocessing as mp
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from pathlib import Path

print(f'PyTorch: {torch.__version__}')
print(f'MPS available: {torch.backends.mps.is_available()}')
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Using device: {device}')

PyTorch: 2.10.0
MPS available: True
Using device: mps


In [2]:
# Verify FerretNet pretrained model loads
from ferretnet.ferret import Ferret
from ferretnet.lpd import get_lpd_dict

lpd_dict = get_lpd_dict()
model = Ferret(in_channels=3, num_classes=1, dim=96, depths=[2,2],
               lpd_func='median', window_size=3, lpd_dict=lpd_dict)
state_dict = torch.load('weights/ferretnet-b-median-3.pth', map_location=device, weights_only=True)
# Weights may be nested under 'model' key
if isinstance(state_dict, dict) and 'model' in state_dict:
    state_dict = state_dict['model']
model.load_state_dict(state_dict)
model.to(device).eval()

# Test forward pass
x = torch.randn(1, 3, 256, 256).to(device)
out = model(x)
prob = torch.sigmoid(out)
print(f'FerretNet loaded! Output shape: {out.shape}, Sample prob: {prob.item():.4f}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

FileNotFoundError: [Errno 2] No such file or directory: 'weights/ferretnet-b-median-3.pth'

In [ ]:
# Verify RetinaFace loads
from retinaface import RetinaFaceDetector

detector = RetinaFaceDetector(weights_path='weights/Resnet50_Final.pth', device=device)

# Test on a sample image
test_img_path = '/Users/MAC/facenew/retina_project/Pytorch_Retinaface/curve/test.jpg'
if Path(test_img_path).exists():
    test_img = Image.open(test_img_path).convert('RGB')
    faces = detector.detect(test_img)
    face_crops = detector.detect_faces(test_img, crop_size=256, margin=0.20)
    print(f'RetinaFace loaded! Detected {len(faces)} faces, {len(face_crops)} crops')
    if face_crops:
        plt.imshow(face_crops[0])
        plt.title('First face crop (256x256)')
        plt.axis('off')
        plt.show()
else:
    print('Test image not found, skipping detection test')

In [ ]:
# Verify DualBranchFerretNet
from ferretnet.dual_branch import DualBranchFerretNet
from configs.base_config import FusionConfig

dual_model = DualBranchFerretNet('weights/ferretnet-b-median-3.pth', FusionConfig())
dual_model.to(device).eval()

face_input = torch.randn(2, 3, 256, 256).to(device)
full_input = torch.randn(2, 3, 256, 256).to(device)
face_logits, full_logits = dual_model(face_input, full_input)
print(f'Dual-branch forward pass OK!')
print(f'Face logits: {face_logits.shape}, Full logits: {full_logits.shape}')
print(f'Total params: {sum(p.numel() for p in dual_model.parameters()):,}')

## 2. Data Download & Face Crop Precomputation

In [ ]:
# Download GRAVEX-200K (requires ~/.kaggle/kaggle.json)
# Uncomment to run:
# !cd /Users/MAC/Desktop/Projects/AI_COMPUTER_VISION && python scripts/download_data.py

# Check if data exists
manifests = Path('/Users/MAC/Desktop/Projects/AI_COMPUTER_VISION/data_raw/manifests')
if manifests.exists():
    for csv_file in manifests.glob('*.csv'):
        with open(csv_file) as f:
            n = sum(1 for _ in f) - 1  # minus header
        print(f'{csv_file.name}: {n} samples')
else:
    print('No manifests found. Run download_data.py first.')

In [ ]:
# Precompute face crops (run AFTER download_data.py)
# This takes ~1 hour for the full dataset
# Uncomment to run:
# !cd /Users/MAC/Desktop/Projects/AI_COMPUTER_VISION && python scripts/precompute_face_crops.py --limit 1000

# Check face crop manifests
for split in ['train', 'val', 'test']:
    p = manifests / f'{split}_with_faces.csv'
    if p.exists():
        with open(p) as f:
            n = sum(1 for _ in f) - 1
        print(f'{split}_with_faces.csv: {n} samples')
    else:
        print(f'{split}_with_faces.csv: NOT FOUND - run precompute_face_crops.py')

## 3. Augmentation Report

In [ ]:
from data.transforms import (
    JPEGCompression, RandomInterpolationResize, GammaCorrection,
    RandomBlur, RandomSharpen, get_train_transforms, get_val_transforms
)
import torchvision.transforms as T

# Load a sample image for augmentation demo
sample_paths = list(Path('/Users/MAC/Desktop/Projects/AI_COMPUTER_VISION/data_raw/gravex_200k').rglob('*.jpg'))[:2]
if not sample_paths:
    # Fallback to any available image
    sample_paths = [Path('/Users/MAC/facenew/retina_project/Pytorch_Retinaface/curve/test.jpg')]

sample_img = Image.open(sample_paths[0]).convert('RGB').resize((256, 256))
print(f'Using sample: {sample_paths[0].name}, size: {sample_img.size}')

In [ ]:
def show_augmentation(original, augmented, title, description):
    """Show before/after + histograms for an augmentation."""
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    
    # Images
    axes[0, 0].imshow(original)
    axes[0, 0].set_title('Before')
    axes[0, 0].axis('off')
    
    if isinstance(augmented, Image.Image):
        axes[0, 1].imshow(augmented)
    else:
        axes[0, 1].imshow(augmented.permute(1, 2, 0).numpy().clip(0, 1))
    axes[0, 1].set_title('After')
    axes[0, 1].axis('off')
    
    # Histograms
    orig_arr = np.array(original)
    aug_arr = np.array(augmented) if isinstance(augmented, Image.Image) else (augmented.permute(1,2,0).numpy()*255).astype(np.uint8)
    for c, color in enumerate(['red', 'green', 'blue']):
        axes[1, 0].hist(orig_arr[:,:,c].flatten(), bins=50, alpha=0.5, color=color, label=f'Ch {c}')
        axes[1, 1].hist(aug_arr[:,:,c].flatten(), bins=50, alpha=0.5, color=color, label=f'Ch {c}')
    axes[1, 0].set_title('Before Histogram')
    axes[1, 0].set_xlabel('Pixel Value')
    axes[1, 1].set_title('After Histogram')
    axes[1, 1].set_xlabel('Pixel Value')
    
    plt.tight_layout()
    plt.show()
    print(f'Description: {description}')
    print()

In [ ]:
# 1. JPEG Compression
jpeg = JPEGCompression(quality_range=(30, 40), probability=1.0)  # low quality to make effect visible
show_augmentation(sample_img, jpeg(sample_img),
    '1. JPEG Compression',
    'Changes: Introduces DCT block artifacts, changes noise characteristics. '
    'Why: Prevents model from learning compression artifacts as proxy for "real". '
    'Failure: Quality <50 destroys too much detail; hurts small face crops.')

In [ ]:
# 2. Rotation
rotated = T.RandomRotation(degrees=15)(sample_img)
show_augmentation(sample_img, rotated,
    '2. Rotation (±15°)',
    'Changes: Geometry — shifts pixel positions, introduces black triangles at corners. '
    'Why: Simulates handheld camera tilt, minor misalignment. '
    'Failure: Large rotations (>30°) break face structure for the face branch.')

In [ ]:
# 3. Horizontal Flip
flipped = T.RandomHorizontalFlip(p=1.0)(sample_img)
show_augmentation(sample_img, flipped,
    '3. Horizontal Flip',
    'Changes: Geometry — mirrors left/right. '
    'Why: Faces can face either direction; doubles effective training data. '
    'Failure: Vertical flip breaks gravity cues — horizontal only.')

In [ ]:
# 4. Random Crop
cropped = T.RandomResizedCrop(256, scale=(0.7, 1.0))(sample_img)
show_augmentation(sample_img, cropped,
    '4. Random Resized Crop (scale 0.7-1.0)',
    'Changes: Geometry — removes border regions, simulates different framing/zoom. '
    'Why: Simulates different camera zoom levels and compositions. '
    'Failure: Aggressive crops (scale <0.5) may remove the face entirely.')

In [ ]:
# 5. Random Interpolation Resize
resize_aug = RandomInterpolationResize((256, 256))
# Demo by resizing to 128 then back to show artifact differences
small = sample_img.resize((128, 128), Image.NEAREST)
back_nearest = small.resize((256, 256), Image.NEAREST)
back_bilinear = small.resize((256, 256), Image.BILINEAR)
back_lanczos = small.resize((256, 256), Image.LANCZOS)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(sample_img); axes[0].set_title('Original 256'); axes[0].axis('off')
axes[1].imshow(back_nearest); axes[1].set_title('NEAREST'); axes[1].axis('off')
axes[2].imshow(back_bilinear); axes[2].set_title('BILINEAR'); axes[2].axis('off')
axes[3].imshow(back_lanczos); axes[3].set_title('LANCZOS'); axes[3].axis('off')
plt.suptitle('5. Resize Interpolation Comparison (128→256)', fontweight='bold')
plt.tight_layout()
plt.show()
print('Changes: Each interpolation introduces different artifacts (blocky/soft/ringing).')
print('Why: Real images come from diverse sources with unknown resize history.')
print('Failure: NEAREST on small crops loses too much detail.')

In [ ]:
# 6. Blur Variants
blur = RandomBlur(p_gaussian=1.0, p_median=0, p_bilateral=0)
gaussian = blur(sample_img)
blur2 = RandomBlur(p_gaussian=0, p_median=1.0, p_bilateral=0)
median = blur2(sample_img)
blur3 = RandomBlur(p_gaussian=0, p_median=0, p_bilateral=1.0)
bilateral = blur3(sample_img)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(sample_img); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(gaussian); axes[1].set_title('Gaussian'); axes[1].axis('off')
axes[2].imshow(median); axes[2].set_title('Median'); axes[2].axis('off')
axes[3].imshow(bilateral); axes[3].set_title('Bilateral'); axes[3].axis('off')
plt.suptitle('6. Blur Variants', fontweight='bold')
plt.tight_layout()
plt.show()
print('Gaussian: smooths all frequencies (camera shake/defocus).')
print('Median: removes salt-and-pepper noise, preserves edges (sensor cleanup).')
print('Bilateral: smooths while preserving edges (phone beauty filter).')
print('Failure: Strong blur (kernel>5) destroys LPD texture patterns.')

In [ ]:
# 7. Sharpening
sharp = RandomSharpen(probability=1.0)(sample_img)
show_augmentation(sample_img, sharp,
    '7. Sharpening (UnsharpMask)',
    'Changes: Enhances high-frequency edges and texture detail. '
    'Why: Simulates user post-processing; makes model focus on texture beyond smoothness. '
    'Failure: Over-sharpening creates halos that look like AI artifacts (false positives).')

In [ ]:
# 8. Histogram Equalization
equalized = T.RandomEqualize(p=1.0)(sample_img)
show_augmentation(sample_img, equalized,
    '8. Histogram Equalization',
    'Changes: Intensity distribution — redistributes pixel values across full range. '
    'Why: Normalizes brightness across cameras and AI generators. '
    'Failure: Can wash out subtle tonal gradients that distinguish AI skin textures.')

In [ ]:
# 9. Random Erasing / Cutout
import torchvision.transforms.functional as TF

tensor_img = T.ToTensor()(sample_img)
# Apply RandomErasing deterministically for demo
erased = T.RandomErasing(p=1.0, scale=(0.05, 0.15), ratio=(0.3, 3.3))(tensor_img)

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes[0, 0].imshow(sample_img); axes[0, 0].set_title('Before'); axes[0, 0].axis('off')
axes[0, 1].imshow(erased.permute(1, 2, 0)); axes[0, 1].set_title('After (Random Erasing)'); axes[0, 1].axis('off')
for c, color in enumerate(['red', 'green', 'blue']):
    axes[1, 0].hist(np.array(sample_img)[:,:,c].flatten(), bins=50, alpha=0.5, color=color)
    arr = (erased.permute(1,2,0).numpy()*255).astype(np.uint8)
    axes[1, 1].hist(arr[:,:,c].flatten(), bins=50, alpha=0.5, color=color)
axes[1, 0].set_title('Before Histogram'); axes[1, 1].set_title('After Histogram')
plt.suptitle('9. Random Erasing (Cutout)', fontweight='bold')
plt.tight_layout()
plt.show()
print('Changes: Occludes random rectangle, fills with random values.')
print('Why: Simulates partial occlusion (objects, watermarks). Prevents spatial overfitting.')
print('Failure: Large erased regions (>20%) may remove the face entirely.')

In [ ]:
# 10. Color Jitter + Gamma
jittered = T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.08)(sample_img)
gamma_corrected = GammaCorrection(gamma_range=(0.5, 0.5), probability=1.0)(sample_img)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(sample_img); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(jittered); axes[1].set_title('Color Jitter'); axes[1].axis('off')
axes[2].imshow(gamma_corrected); axes[2].set_title('Gamma=0.5'); axes[2].axis('off')
plt.suptitle('10. Color Jitter + Gamma Correction', fontweight='bold')
plt.tight_layout()
plt.show()
print('Color Jitter: Shifts brightness, contrast, saturation, hue.')
print('Gamma: img = img^gamma — nonlinear brightness adjustment.')
print('Why: Simulates diverse lighting, white balance, screen calibration.')
print('Failure: Hue shifts >0.15 create unrealistic colors.')

In [ ]:
# Augmentation Summary Table
import pandas as pd

augmentations = [
    ('JPEG Compression', 'Intensity/Noise', '0.50', 'quality=65-100'),
    ('Rotation', 'Geometry', '1.00', 'degrees=±15'),
    ('Horizontal Flip', 'Geometry', '0.50', 'horizontal only'),
    ('Random Crop', 'Geometry', '1.00', 'scale=0.7-1.0'),
    ('Random Interp Resize', 'Intensity', '1.00', 'BILINEAR/BICUBIC/NEAREST/LANCZOS'),
    ('Random Blur', 'Noise', '0.35', 'Gaussian/Median/Bilateral'),
    ('Sharpening', 'Intensity', '0.15', 'UnsharpMask r=2 p=150'),
    ('Histogram Equalize', 'Intensity', '0.10', 'full range equalization'),
    ('Random Erasing', 'Geometry', '0.15', 'scale=0.02-0.15'),
    ('Color Jitter + Gamma', 'Intensity', '1.00', 'b=0.3,c=0.3,s=0.2,h=0.08,gamma=0.8-1.2'),
]

df = pd.DataFrame(augmentations, columns=['Augmentation', 'Category', 'Probability', 'Parameters'])
df

## 4. Training

Run fine-tuning with resume support. Training state is saved every epoch.
If interrupted, re-run the same cell and it will resume from the last checkpoint.

In [ ]:
# Check if face crop manifests exist (required for training)
manifests_dir = Path('/Users/MAC/Desktop/Projects/AI_COMPUTER_VISION/data_raw/manifests')
train_manifest = manifests_dir / 'train_with_faces.csv'
val_manifest = manifests_dir / 'val_with_faces.csv'

if train_manifest.exists() and val_manifest.exists():
    import csv
    with open(train_manifest) as f:
        n_train = sum(1 for _ in f) - 1
    with open(val_manifest) as f:
        n_val = sum(1 for _ in f) - 1
    print(f'Train: {n_train} samples, Val: {n_val} samples')
    print('Ready to train!')
else:
    print('MISSING: Face crop manifests not found.')
    print('Run these first:')
    print('  python scripts/download_data.py')
    print('  python scripts/precompute_face_crops.py')

In [ ]:
# Training — Run fine-tuning
# This cell starts the training pipeline.
# To resume after interruption, simply re-run this cell.

import pytorch_lightning as pl
from torch.utils.data import DataLoader
from configs.base_config import ProjectConfig, FerretNetConfig, TrainConfig, FusionConfig
from data.transforms import get_train_transforms, get_val_transforms
from data.dual_branch_dataset import DualBranchDataset
from ferretnet.lightning_module import FerretNetLightning
from training.callbacks import ValMetricsCSV

proj = ProjectConfig()
train_cfg = TrainConfig()

# Find latest checkpoint for resume
ckpt_path = None
last_ckpt = list(Path(proj.lightning_logs).glob('**/last.ckpt'))
if last_ckpt:
    ckpt_path = str(sorted(last_ckpt, key=lambda p: p.stat().st_mtime)[-1])
    print(f'Resuming from: {ckpt_path}')

# Datasets
train_transform = get_train_transforms(256)
val_transform = get_val_transforms(256)

train_dataset = DualBranchDataset(
    manifest_csv=str(proj.data_raw / 'manifests' / 'train_with_faces.csv'),
    face_transform=train_transform, full_transform=train_transform, face_size=256,
)
val_dataset = DualBranchDataset(
    manifest_csv=str(proj.data_raw / 'manifests' / 'val_with_faces.csv'),
    face_transform=val_transform, full_transform=val_transform, face_size=256,
)

train_loader = DataLoader(train_dataset, batch_size=train_cfg.batch_size, shuffle=True,
                          num_workers=train_cfg.num_workers, persistent_workers=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=train_cfg.batch_size, shuffle=False,
                        num_workers=train_cfg.num_workers, persistent_workers=True)

# Model
model = FerretNetLightning(
    ferretnet_config=FerretNetConfig(),
    train_config=train_cfg,
    fusion_config=FusionConfig(),
    pretrained_path=str(proj.ferretnet_weights),
)

# Callbacks
callbacks = [
    pl.callbacks.ModelCheckpoint(monitor='val_fused_acc', mode='max', save_top_k=3,
                                 filename='ferretnet-{epoch:02d}-{val_fused_acc:.4f}',
                                 save_last=True),
    pl.callbacks.EarlyStopping(monitor='val_fused_acc', patience=train_cfg.early_stopping_patience, mode='max'),
    pl.callbacks.LearningRateMonitor(logging_interval='epoch'),
    ValMetricsCSV(csv_path=str(proj.results_dir / 'val_metrics.csv')),
]

trainer = pl.Trainer(
    accelerator='mps', devices=1,
    max_epochs=train_cfg.max_epochs,
    gradient_clip_val=train_cfg.gradient_clip_val,
    precision=train_cfg.precision,
    callbacks=callbacks,
    default_root_dir=str(proj.lightning_logs),
)

print(f'Starting training: {train_cfg.max_epochs} epochs, batch={train_cfg.batch_size}, lr={train_cfg.learning_rate}')
trainer.fit(model, train_loader, val_loader, ckpt_path=ckpt_path)

In [ ]:
# Plot training metrics from CSV
import pandas as pd

metrics_csv = proj.results_dir / 'val_metrics.csv'
if metrics_csv.exists():
    df = pd.read_csv(metrics_csv)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    axes[0].plot(df['epoch'], df['train_loss'], label='Train Loss')
    axes[0].plot(df['epoch'], df['val_loss'], label='Val Loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend()
    axes[0].set_title('Loss')
    
    axes[1].plot(df['epoch'], df['val_face_acc'], label='Face')
    axes[1].plot(df['epoch'], df['val_full_acc'], label='Full')
    axes[1].plot(df['epoch'], df['val_fused_acc'], label='Fused')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy'); axes[1].legend()
    axes[1].set_title('Validation Accuracy')
    
    axes[2].plot(df['epoch'], df['lr'])
    axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Learning Rate')
    axes[2].set_title('Learning Rate Schedule')
    
    plt.tight_layout()
    plt.show()
else:
    print('No val_metrics.csv found yet.')

## 5. Evaluation on Test Set

In [ ]:
# Find best checkpoint
checkpoints = sorted(Path(proj.lightning_logs).glob('**/*.ckpt'))
best_ckpt = None
for ckpt in checkpoints:
    if 'last' not in ckpt.name:
        best_ckpt = str(ckpt)
        break
if not best_ckpt and checkpoints:
    best_ckpt = str(checkpoints[-1])

if best_ckpt:
    print(f'Evaluating: {best_ckpt}')
else:
    print('No checkpoint found for evaluation.')

In [ ]:
# Run evaluation
if best_ckpt:
    from evaluation.evaluate import main as eval_main
    import sys
    sys.argv = ['evaluate.py', '--checkpoint', best_ckpt]
    eval_main()

## 6. Inference Demo

In [ ]:
# Test on sample images
if best_ckpt:
    from inference.pipeline import FerretNetPipeline
    
    pipeline = FerretNetPipeline(
        checkpoint_path=best_ckpt,
        retinaface_weights='weights/Resnet50_Final.pth',
        device=device,
    )
    
    # Test on a few images
    test_images = list(Path('/Users/MAC/Desktop/Projects/AI_COMPUTER_VISION/data_raw/gravex_200k').rglob('*.jpg'))[:5]
    if not test_images:
        test_images = [Path('/Users/MAC/facenew/retina_project/Pytorch_Retinaface/curve/test.jpg')]
    
    for img_path in test_images[:3]:
        result = pipeline.predict(str(img_path))
        img = Image.open(img_path).resize((256, 256))
        plt.figure(figsize=(3, 3))
        plt.imshow(img)
        plt.title(f'{result["label"]} ({result["fused_score"]:.3f})')
        plt.axis('off')
        plt.show()
        print(f'  Faces: {result["faces_detected"]}, '
              f'Face scores: {[f"{s:.3f}" for s in result["face_scores"]]}, '
              f'Full: {result["full_score"]:.3f}, Fused: {result["fused_score"]:.3f}')